In [5]:
import polars as pl

In [3]:
# create project root dir 
from pathlib import Path
project_root = Path.cwd().parent.resolve()




In [ ]:
# reade cleaned budget parquet

data_dir = "data/processed"
df = pl.read_parquet(project_root / data_dir / "budget_cleaned.parquet")

# find unique values in type and fiscal_year columns using polars
print("Unique values in 'type' column:")
print(df.select(pl.col("type").unique()).to_series().to_list())
print("\nUnique values in 'fiscal_year' column:")
print(df.select(pl.col("fiscal_year").unique()).to_series().to_list())

In [15]:
dim_df = df = pl.read_parquet(project_root / data_dir / "budget_dim.parquet")

# # export this as csv

dim_df.write_csv(project_root / data_dir /"budget_dim.csv")

In [19]:
# load cost pool mappings YAML and join to subobject codes by code
import yaml

mapping_path = project_root / "configs" / "cost_pool_mappings.yaml"

with open(mapping_path, "r", encoding="utf-8") as f:
    mapping_cfg = yaml.safe_load(f) or {}

lookup = mapping_cfg.get("mappings", {})

subobject_df = pl.read_csv(project_root / "data" / "processed" / "subobjects.csv")

# normalize join key to 4-digit code so values like 869 match YAML key 0869
subobject_df = subobject_df.with_columns(
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4)
 )

mapping_rows = [
    {
        "comptroller_subobject_code": str(code).strip().zfill(4),
        "cost_pool": attrs.get("cost_pool"),
        "cost_sub_pool": attrs.get("cost_sub_pool"),
    }
    for code, attrs in lookup.items()
    if isinstance(attrs, dict)
]

mapping_df = pl.DataFrame(mapping_rows)

joined_df = subobject_df.join(mapping_df, on="comptroller_subobject_code", how="left")

print("Rows:", joined_df.height)
print("Mapped rows:", joined_df.filter(pl.col("cost_pool").is_not_null()).height)
print("Unmapped rows:", joined_df.filter(pl.col("cost_pool").is_null()).height)

joined_df.select(
    [   "object_code",
        "object_name",
        "comptroller_subobject_code",
        "comptroller_subobject_name",
        "cost_pool",
        "cost_sub_pool",
    ]
).head(20)

Rows: 371
Mapped rows: 371
Unmapped rows: 0


object_code,object_name,comptroller_subobject_code,comptroller_subobject_name,cost_pool,cost_sub_pool
i64,str,str,str,str,str
1,"""Salaries, Wages and Fringe Ben…","""0101""","""Regular Earnings""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0102""","""Additional Assistance""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0104""","""Overtime Earnings""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0105""","""Shift Differential""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0110""","""Miscellaneous Adjustments""","""Staffing""","""Internal Labor"""
…,…,…,…,…,…
1,"""Salaries, Wages and Fringe Ben…","""0161""","""Employees' Retirement""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0162""","""Employees' Pension System""","""Staffing""","""Internal Labor"""
1,"""Salaries, Wages and Fringe Ben…","""0163""","""Teachers' Retirement System""","""Staffing""","""Internal Labor"""


In [20]:
# save csv in data/processed
joined_df.write_csv(project_root / data_dir / "subobjects.csv")

In [21]:
# join subprogram with tower classifications on organization_sub_code
subprogram_path = project_root / "data" / "processed" / "subprograms.csv"
tower_cls_path = project_root / "data" / "output" / "tower_classifications.csv"

subprogram_df = pl.read_csv(subprogram_path)
tower_df = pl.read_csv(tower_cls_path, encoding="utf8-lossy")

join_key = "organization_sub_code"

subprogram_df = subprogram_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)
tower_df = tower_df.with_columns(
    pl.col(join_key).cast(pl.Utf8).str.strip_chars()
)

# keep only join key + last 4 columns from tower classifications
col_needed = tower_df.columns[-4:]
tower_keep_cols = [join_key, *col_needed]

joined_subprogram_tower_df = subprogram_df.join(
    tower_df.select(tower_keep_cols),
    on=join_key,
    how="left",
)

print("Rows:", joined_subprogram_tower_df.height)
print("Tower columns appended:", col_needed)

joined_subprogram_tower_df.head(20)


Rows: 4978
Tower columns appended: ['it_designation', 'tower', 'sub_tower', 'confidence']


organization_sub_code,organization_code,agency_code,agency_name,unit_code,unit_name,program_code,program_name,subprogram_code,subprogram_name,description,category,category_title,is_it,it_designation,shadow_it_reason,it_designation_right,tower,sub_tower,confidence
str,str,str,str,str,str,i64,str,str,str,str,i64,str,bool,str,str,str,str,str,f64
"""A15_O00_01_1BSL""","""A15_O00_01""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",1,"""Disparity Grants""","""1BSL""","""Disparity Grants""","""Section 16-501 of the Local Go…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_02_2BSL""","""A15_O00_02""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",2,"""Teacher Retirement Supplementa…","""2BSL""","""Teacher Retirement Supplementa…","""Section 16-503 of the Local Go…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_03_3BSL""","""A15_O00_03""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",3,"""Admissions and Amusement Tax D…","""3BSL""","""Admissions and Amusement Tax D…","""The grants in this program rep…",10,"""Other""",false,null,null,null,null,null,null
"""A15_O00_05_5BSL""","""A15_O00_05""","""A15""","""Payments to Civil Divisions of…","""O00""","""Payments to Civil Divisions of…",5,"""Cannabis Sales Tax Distributio…","""5BSL""","""Cannabis Sales Tax Distributio…","""This program represents revenu…",10,"""Other""",false,null,null,null,null,null,null
"""B75_A01_01_0000""","""B75_A01_01""","""B75""","""Legislative Branch""","""A01""","""Legislative Branch""",1,"""Senate""","""0""","""Senate""","""The Senate is composed of 47 S…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""C00_A00_03_C003""","""C00_A00_03""","""C00""","""Judiciary""","""A00""","""Judiciary""",3,"""Circuit Court Judges""","""C003""","""Circuit Court Masters Child Su…","""The Circuit Courts for Marylan…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
"""C00_A00_03_D003""","""C00_A00_03""","""C00""","""Judiciary""","""A00""","""Judiciary""",3,"""Circuit Court Judges""","""D003""","""Law Clerks""","""The Circuit Courts for Marylan…",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null
"""C00_A00_04_B004""","""C00_A00_04""","""C00""","""Judiciary""","""A00""","""Judiciary""",4,"""District Court""","""B004""","""District Court""","""Article IV, Section 1, of the …",8,"""Legislative, Judicial, Legal""",false,null,null,null,null,null,null


In [22]:
joined_subprogram_tower_df.write_csv(project_root / "data" / "processed" / "subprograms.csv")

In [ ]:
# join budget parquet with IT subprograms and subobject mappings

budget_path = project_root / "data" / "processed" / "budget_cleaned.parquet"
budget_df = pl.read_parquet(budget_path)

it_subprograms_path = project_root / "data" / "processed" / "it_subprogram.csv"
it_subprograms_df = pl.read_csv(it_subprograms_path)

subobject_codes_path = project_root / "data" / "processed" / "subobjects.csv"
subobject_codes_df = pl.read_csv(subobject_codes_path)

# normalize join key to 4-digit strings so 869 and 0869 match
budget_df = budget_df.with_columns([
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4).alias("comptroller_subobject_code")
])

subobject_codes_df = subobject_codes_df.with_columns([
    pl.col("comptroller_subobject_code").cast(pl.Utf8).str.strip_chars().str.zfill(4).alias("comptroller_subobject_code")
])

budget_columns = ['fiscal_year',
 'organization_sub_code',                 
 'comptroller_subobject_code',               
 'agency_subobject_code',
 'agency_subobject_name',
 'fund_type_name',
 'budget',
 'type',
 'organization_code',
 'category',
 'category_title']

joined_df = budget_df.select(budget_columns
                             ).join(
    it_subprograms_df,
    on="organization_sub_code",
    how="left"
).join(
    subobject_codes_df,
    on="comptroller_subobject_code",
    how="left"
)

In [46]:
joined_df.columns

['fiscal_year',
 'organization_sub_code',
 'comptroller_subobject_code',
 'agency_subobject_code',
 'agency_subobject_name',
 'fund_type_name',
 'budget',
 'type',
 'organization_code',
 'category',
 'category_title',
 'agency_code',
 'agency_name',
 'unit_code',
 'unit_name',
 'program_code',
 'program_name',
 'subprogram_code',
 'subprogram_name',
 'organization_code_right',
 'description',
 'is_IT',
 'IT_designination',
 'tower',
 'sub_tower',
 'confidence',
 'object_code',
 'object_name',
 'comptroller_subobject_name',
 'cost_pool',
 'cost_sub_pool']

In [55]:
joined_df.write_csv(project_root / "data" / "enriched" / "budget_enriched.csv")
joined_df.write_parquet(project_root / "data" / "enriched" / "budget_enriched.parquet")